In [1]:
pip install mlflow dagshub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 71.9 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 90.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!pip install nltk

### Set Mlflow

In [3]:
import dagshub
dagshub.init(repo_owner='iamdebasishdas123', repo_name='YouTube-Mood-Tracker', mlflow=True)
import mlflow

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=d726c502-9a54-4e3c-b366-36526cf52bc3&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=46b4f0d2db28c292f359edc3704e9a243dc5b305d7aced57f53ae25a26723dfe




Accessing as iamdebasishdas123

Initialized MLflow to track repo "iamdebasishdas123/YouTube-Mood-Tracker"

Repository iamdebasishdas123/YouTube-Mood-Tracker initialized!

## Load Dataset

In [4]:
import numpy as np
import pandas as pd

In [5]:
# Load the data
def load_data():
    df = pd.read_csv('https://raw.githubusercontent.com/Himanshu-1703/reddit-sentiment-analysis/refs/heads/main/data/reddit.csv')
    return df
df = load_data()
df.head()

,clean_comment,category
0,family mormon have never tried explain them t...,1
1,buddhism has very much lot compatible with chr...,1
2,seriously don say thing first all they won get...,-1
3,what you have learned yours and only yours wha...,0
4,for your own benefit you may want read living ...,1


## 1. Preprocessing Data

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37249 entries, 0 to 37248
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   clean_comment  37149 non-null  object
 1   category       37249 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 582.1+ KB


In [7]:
df['category'].value_counts()

,count
category,
1,15830
0,13142
-1,8277


In [8]:
df.duplicated().sum()

np.int64(449)

- Problem of Database
1. backslash in the data
2. Null data arund 100 rows
3. 449 duplicated data
3. Imbalanced data 

##### 1.1 Remove NA data

In [9]:
df.dropna(inplace=True)

##### 1.2 Remove Duplicate

In [10]:
df.drop_duplicates(inplace=True)

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 36799 entries, 0 to 37248
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   clean_comment  36799 non-null  object
 1   category       36799 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 862.5+ KB


##### 1.3 Clean Comment

In [12]:
df = df[~(df['clean_comment'].str.strip() == '')]

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 36793 entries, 0 to 37248
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   clean_comment  36793 non-null  object
 1   category       36793 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 862.3+ KB


In [14]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [15]:
# Ensure necessary NLTK data is downloaded
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

##### 1.4 Preprocessing Step

In [16]:
# Define the preprocessing function
def preprocess_comment(comment):
    # Convert to lowercase
    comment = comment.lower()

    # Remove trailing and leading whitespaces
    comment = comment.strip()

    # Remove newline characters
    comment = re.sub(r'\n', ' ', comment)

    # Remove non-alphanumeric characters, except punctuation
    comment = re.sub(r'[^A-Za-z0-9\s!?.,]', '', comment)

    # Remove stopwords but retain important ones for sentiment analysis
    stop_words = set(stopwords.words('english')) - {'not', 'but', 'however', 'no', 'yet'}
    comment = ' '.join([word for word in comment.split() if word not in stop_words])

    # Lemmatize the words
    lemmatizer = WordNetLemmatizer()
    comment = ' '.join([lemmatizer.lemmatize(word) for word in comment.split()])

    return comment

In [17]:
# Apply the preprocessing function to the 'clean_comment' column
df['clean_comment'] = df['clean_comment'].apply(preprocess_comment)

## 2. Baseline Model Building

In [22]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Step 1: Vectorize the comments using Bag of Words (CountVectorizer)
vectorizer = CountVectorizer(max_features=5000)  # Bag of Words model with a limit of 5000 features

In [ ]:
X = vectorizer.fit_transform(df['clean_comment']).toarray()
y = df['category']  # Assuming 'sentiment' is the target variable (0 or 1 for binary classification)

In [ ]:
X

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [ ]:
X.shape

(36793, 5000)

In [ ]:

y = y.map({-1: 0, 0: 1, 1: 2})


In [ ]:
y.shape

(36793,)

In [ ]:
# Set or create an experiment
mlflow.set_experiment("Test Experiment Youtube Mood Tracker")

<Experiment: artifact_location='mlflow-artifacts:/a919a8ba2f0a451c8fdec499a9ec20b4', creation_time=1778520588352, experiment_id='0', last_update_time=1778520588352, lifecycle_stage='active', name='Test Experiment Youtube Mood Tracker', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [ ]:
import seaborn as sns

In [ ]:
# Step 1: Split the data into training and testing sets (80% train, 20% test)
import matplotlib.pyplot as plt
import mlflow
import mlflow.xgboost  # Crucial for logging XGBoost correctly
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# Step 1: Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Step 2: Define and train an XGBoost baseline model
with mlflow.start_run() as run:
    # Metadata tags
    mlflow.set_tag("mlflow.runName", "XGBoost_Baseline_TrainTestSplit")
    mlflow.set_tag("experiment_type", "baseline")
    mlflow.set_tag("model_type", "XGBClassifier")
    mlflow.set_tag(
        "description",
        "Baseline XGBoost sentiment model using Bag of Words (BoW)",
    )

    # Log vectorizer params
    mlflow.log_param("vectorizer_type", "CountVectorizer")
    mlflow.log_param("vectorizer_max_features", vectorizer.max_features)

    # Model Hyperparameters
    params = {
        "n_estimators": 350,
        "max_depth": 12,
        "learning_rate": 0.1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "random_state": 42,
        "eval_metric": "logloss",
    }
    mlflow.log_params(params)  # Dictionary logging keeps code clean

    # Train model
    model = XGBClassifier(**params)
    model.fit(X_train, y_train)

    # Evaluate
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    mlflow.log_metric("accuracy", accuracy)

    # Classification report logic
    classification_rep = classification_report(y_test, y_pred, output_dict=True)
    for label, metrics in classification_rep.items():
        if isinstance(metrics, dict):
            # Formats keys cleanly as "class_0_precision", "macro_avg_f1-score"
            clean_label = label.replace(" ", "_")
            for metric, value in metrics.items():
                mlflow.log_metric(f"{clean_label}_{metric}", value)

    # Confusion Matrix
    conf_matrix = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Confusion Matrix")

    # Consistent local paths to prevent file-not-found errors
    plt.savefig("confusion_matrix.png")
    mlflow.log_artifact("confusion_matrix.png")
    plt.close()  # Close the plot to free memory

    # Native XGBoost logging
    mlflow.xgboost.log_model(model, "xgboost_model")

    # Save data snippet
    df.to_csv("dataset.csv", index=False)
    mlflow.log_artifact("dataset.csv")

print(f"Accuracy: {accuracy}")


2026/05/18 17:22:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost_Baseline_TrainTestSplit at: https://dagshub.com/iamdebasishdas123/YouTube-Mood-Tracker.mlflow/#/experiments/0/runs/d41936a8fcde4c67b18cfb94565807ff
🧪 View experiment at: https://dagshub.com/iamdebasishdas123/YouTube-Mood-Tracker.mlflow/#/experiments/0
Accuracy: 0.8498437287674956


In [ ]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.84      0.69      0.76      1650
           1       0.82      0.97      0.89      2555
           2       0.89      0.84      0.86      3154

    accuracy                           0.85      7359
   macro avg       0.85      0.83      0.83      7359
weighted avg       0.85      0.85      0.85      7359



### Run Experiments


##### 1.1 BOW or TFIDF 

- Bag of Word(BOW)
1. It is a NLP technique to create embedding
2. Create unique collection of words from the all documents.
3. Count how many times each word from the master vocabulary appears in a specific document.
4. Represent each document as an array (vector) of these word counts.

- Example
Document 1: "The dog barked at the cat.

Document 2: "The cat sat on the mat."First

compile the unique vocabulary: ['the', 'dog', 'barked', 'at', 'cat', 'sat', 'on', 'mat'] (Total length: 8).

Using BoW, we represent the documents as numerical frequency vectors:

Document 1: \([2, 1, 1, 1, 1, 0, 0, 0]\)

Document 2: \([2, 0, 0, 0, 1, 1, 1, 1]\)

In [18]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

def compute_bow(corpus, max_features=None):
    """
    Converts a text corpus into a Bag of Words matrix.
    """
    # Initialize the vectorizer
    vectorizer = CountVectorizer(max_features=max_features)
    
    # Learn vocabulary and transform text to sparse matrix
    bow_sparse = vectorizer.fit_transform(corpus)
    
    # Convert to a readable DataFrame with word labels
    feature_names = vectorizer.get_feature_names_out()
    bow_df = pd.DataFrame(bow_sparse.toarray(), columns=feature_names)
    
    return bow_df, vectorizer


- TF-IDF (Term Frequency-Inverse Document Frequency) : TF-IDF (Term Frequency-Inverse Document Frequency) is a foundational natural language processing algorithm that measures how important a word is to a specific document within a collection of text [1, 2]. It significantly improves upon the classic Bag of Words model because it does not just count how often words appear; it penalises words that show up everywhere (like "the", "is", or "and") and rewards unique words that convey actual meaning.

1. Term Frequency (TF): Measures how frequently a word appears inside a single document.
2. Inverse Document Frequency (IDF): Measures how unique or rare a word is across your entire collection of documents (corpus).

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

def compute_tfidf(corpus, max_features=None):
    """
    Converts a text corpus into a TF-IDF matrix.
    """
    # Initialize the vectorizer
    vectorizer = TfidfVectorizer(max_features=max_features)
    
    # Learn vocabulary and weights, transform text to sparse matrix
    tfidf_sparse = vectorizer.fit_transform(corpus)
    
    # Convert to a readable DataFrame with word labels
    feature_names = vectorizer.get_feature_names_out()
    tfidf_df = pd.DataFrame(tfidf_sparse.toarray(), columns=feature_names)
    
    return tfidf_df, vectorizer


##### 1.2 Model Selection

- **Logistic Regression:** 
- Logistic regression for 3 (or more) label classification is known as Multiclass Classification. Because standard logistic regression is designed for only two outcomes, it relies on two primary approaches to handle three labels: Multinomial Logistic Regression (Softmax) and One-vs-Rest (OvR).
-  It uses the Softmax function to convert raw, continuous model outputs (logits) into a proper probability distribution that sums to 1.0 (or \(100\%\)) across the 3 labels.(Soft Max)
- During prediction, all 3 classifiers evaluate the data point. The model assigns the label of the classifier that returns the highest probability score.(OVR)

In [20]:
from sklearn.linear_model import LogisticRegression
def logistic_model(multi_class='multinomial'):
    model=LogisticRegression(multi_class=multi_class, solver='lbfgs') # or use Muti_class='ovr' for one-vs-rest
    return model

- **SVM :**
* A Support Vector Machine (SVM) is a powerful supervised machine learning algorithm used primarily for classification, regression, and outlier detection. It works by finding an optimal decision boundary (a hyperplane) that separates different categories of data with the maximum possible margin

In [21]:
from sklearn.svm import SVC
# kernel can be 'linear', 'poly', 'rbf', 'sigmoid', 'precomputed' or a callable. Default is 'rbf'.
# gamma can be 'scale' or a float. Default is 'scale'.
# poly degree can be specified if kernel is 'poly'
# linear kernel is often a good choice for text data, but RBF can capture non-linear relationships.
# sigmoid kernel can be useful for certain types of data, but is less common in text classification.

def svm_model(kernel='rbf', C=1.0):
    model=SVC(kernel=kernel, C=C, gamma='scale')
    return model

* **Desision Tree**
* A Decision Tree is a non-parametric supervised learning algorithm used for both classification and regression tasks. It breaks down a dataset into smaller subsets while simultaneously developing an associated decision tree in a flowchart-like structure.

In [22]:
from sklearn.tree import DecisionTreeClassifier
def decision_tree_model(max_depth=10):
    model=DecisionTreeClassifier(max_depth=max_depth, random_state=42)
    return model

* **Random Forest**
* Random Forest is an ensemble machine learning algorithm used for both classification and regression tasks. It operates by constructing a large number of individual decision trees during training and merging their predictions together to get a more accurate and stable result.
* Ensemble Learning: It combines multiple machine learning models (decision trees) to create one stronger, more reliable model.
* Bagging (Bootstrap Aggregating): It trains each decision tree on a random sample of the training data, allowing individual trees to see different subsets of rows.
* Feature Randomness: It forces each tree to split nodes using a random subset of features instead of looking at all available columns.
* Voting/Averaging: For classification tasks, it takes a majority vote from all trees; for regression tasks, it averages the numerical outputs.

In [23]:
from sklearn.ensemble import RandomForestClassifier
def random_forest_model(n_estimators=100, max_depth=10):
    model=RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    return model

* **XGBoost:**
* XGBoost, short for Extreme Gradient Boosting, is an advanced, highly efficient implementation of the gradient boosted decision trees algorithm. It is one of the most popular and dominant machine learning libraries for structured or tabular data, frequently used to win Kaggle competitions and power commercial production systems.
* Gradient Boosting: Unlike Random Forest, which builds independent trees in parallel, XGBoost builds trees sequentially.
* Learning from Mistakes: Each new tree is trained specifically to correct the residual errors (mistakes) made by the previous trees.
* Gradient Descent: It uses a gradient descent optimization algorithm to minimize a specific loss function as it adds new models.

In [24]:
from xgboost import XGBClassifier
def xgboost_model(n_estimators=350, max_depth=12, learning_rate=0.1):
    model=XGBClassifier(n_estimators=n_estimators, max_depth=max_depth, learning_rate=learning_rate, subsample=0.8, colsample_bytree=0.8, random_state=42, eval_metric='logloss')
    return model

#### Experiments with Multiple model and Embedding Model

In [ ]:
# Run grid of experiments: BOW and TF-IDF with multiple feature sizes and models, all logged to MLflow
import os
import time
import mlflow
import mlflow.sklearn
import mlflow.xgboost
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Feature sizes to evaluate
feature_sizes = [4000, 6000, 8000, 10000]
vectorizers = {'bow': CountVectorizer, 'tfidf': TfidfVectorizer}

# Model constructors defined earlier in the notebook (reuse those functions)
model_constructors = {
    'LogisticRegression': lambda: logistic_model(),
    'SVM': lambda: svm_model(),
    'DecisionTree': lambda: decision_tree_model(),
    'RandomForest': lambda: random_forest_model(),
    'XGBoost': lambda: xgboost_model(),
}

results = []
corpus = df['clean_comment'].fillna('').astype(str)
# ensure labels are mapped consistently to 0/1/2
labels = df['category'].map({-1: 0, 0: 1, 1: 2})

# Run experiments
for vec_name, VecClass in vectorizers.items():
    for n_features in feature_sizes:
        print(f'=== Vectorizer: {vec_name}, features: {n_features} ===')
        vectorizer = VecClass(max_features=n_features)
        X = vectorizer.fit_transform(corpus)
        y = labels

        for model_name, make_model in model_constructors.items():
            run_name = f'{model_name}_{vec_name}_{n_features}'
            print('Run:', run_name)
            with mlflow.start_run(run_name=run_name):
                # Tags and params
                mlflow.set_tag('experiment_type', 'grid_search')
                mlflow.set_tag('model_family', model_name)
                mlflow.log_param('vectorizer', vec_name)
                mlflow.log_param('vectorizer_max_features', n_features)
                mlflow.log_param('run_started_at', time.strftime('%Y-%m-%d %H:%M:%S'))

                model = make_model()
                # Try to log model hyperparameters when possible
                try:
                    mlflow.log_params(model.get_params())
                except Exception:
                    pass

                # Split and train
                X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)

                # Metrics
                acc = accuracy_score(y_test, y_pred)
                mlflow.log_metric('accuracy', float(acc))

                crep = classification_report(y_test, y_pred, output_dict=True)
                for label, metrics in crep.items():
                    if isinstance(metrics, dict):
                        clean_label = str(label).replace(' ', '_')
                        for metric, value in metrics.items():
                            try:
                                mlflow.log_metric(f'{clean_label}_{metric}', float(value))
                            except Exception:
                                pass

                # Confusion matrix artifact
                cm = confusion_matrix(y_test, y_pred)
                plt.figure(figsize=(6, 5))
                sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
                plt.xlabel('Predicted')
                plt.ylabel('Actual')
                plt.title(f'Confusion Matrix: {run_name}')
                fname = f'confusion_{run_name}.png'
                plt.savefig(fname)
                plt.close()
                mlflow.log_artifact(fname)

                # Log model artifact (use native logger for xgboost)
                try:
                    if model_name == 'XGBoost':
                        mlflow.xgboost.log_model(model, 'model')
                    else:
                        mlflow.sklearn.log_model(model, 'model')
                except Exception:
                    pass

                # Save a small dataset snapshot for reproducibility
                try:
                    sample_name = f'dataset_snapshot_{run_name}.csv'
                    pd.concat([pd.Series(y_test, name='y_test').reset_index(drop=True), pd.DataFrame(X_test.toarray())], axis=1).to_csv(sample_name, index=False)
                    mlflow.log_artifact(sample_name)
                except Exception:
                    # skip if X_test is large or not convertible to array
                    pass

                # Record summary for later comparison
                results.append({'run': run_name, 'vectorizer': vec_name, 'n_features': n_features, 'model': model_name, 'accuracy': float(acc)})

# Summarize results across all runs
res_df = pd.DataFrame(results)
if not res_df.empty:
    res_df = res_df.sort_values(['model', 'vectorizer', 'n_features'], ascending=[True, True, True]).reset_index(drop=True)
    display(res_df)
else:
    print('No results recorded.')

=== Vectorizer: bow, features: 4000 ===
Run: LogisticRegression_bow_4000


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
2026/05/22 17:58:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/22 17:58:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpick

🏃 View run LogisticRegression_bow_4000 at: https://dagshub.com/iamdebasishdas123/YouTube-Mood-Tracker.mlflow/#/experiments/0/runs/fe9a068c27fa4e2792051da96a813bcf
🧪 View experiment at: https://dagshub.com/iamdebasishdas123/YouTube-Mood-Tracker.mlflow/#/experiments/0
Run: SVM_bow_4000


2026/05/22 18:03:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/22 18:04:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run SVM_bow_4000 at: https://dagshub.com/iamdebasishdas123/YouTube-Mood-Tracker.mlflow/#/experiments/0/runs/77f1b7f77158449c852f1eece05a3dda
🧪 View experiment at: https://dagshub.com/iamdebasishdas123/YouTube-Mood-Tracker.mlflow/#/experiments/0
Run: DecisionTree_bow_4000


...

🏃 View run SVM_bow_6000 at: https://dagshub.com/iamdebasishdas123/YouTube-Mood-Tracker.mlflow/#/experiments/0/runs/19bfc6ffab144c8fa7d13625f571e13e
🧪 View experiment at: https://dagshub.com/iamdebasishdas123/YouTube-Mood-Tracker.mlflow/#/experiments/0
Run: DecisionTree_bow_6000


2026/05/22 18:19:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/22 18:19:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run DecisionTree_bow_6000 at: https://dagshub.com/iamdebasishdas123/YouTube-Mood-Tracker.mlflow/#/experiments/0/runs/5970931fb5e64217862de28e52422368
🧪 View experiment at: https://dagshub.com/iamdebasishdas123/YouTube-Mood-Tracker.mlflow/#/experiments/0
Run: RandomForest_bow_6000


2026/05/22 18:21:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/22 18:21:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RandomForest_bow_6000 at: https://dagshub.com/iamdebasishdas123/YouTube-Mood-Tracker.mlflow/#/experiments/0/runs/d69896ceab79482b98621e024d09b255
🧪 View experiment at: https://dagshub.com/iamdebasishdas123/YouTube-Mood-Tracker.mlflow/#/experiments/0
Run: XGBoost_bow_6000
